# Anchorage Harbor: time-series dataset

From the depth images (`anchorage_dataset.ipynb`) to training samples: K previous maps → the next map.

1. Setup and load
2. Seasons: no sample crosses a winter
3. Flag dredging, per cell
4. Check the flags on maps
5. Area: crop to the core channel
6. Build the samples
7. Save

We use the summer surveys (`IS`), ~52 per year, all covering the whole channel, plus the spring
pre-dredge survey that opens most seasons. Winter surveys (`WS`) are small and patchy.

## 1. Setup and load

`table` is the long table (one row per survey and cell), `images` the same depths as a stack of 2-D maps.
We need both: the dredging rule looks at neighbouring cells, which is easy on a map,
and the result goes back into the table as a new column.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import label

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ehydro/anchorage'
except ImportError:              # not in Colab: the data folder next to the notebook
    DATA_DIR = 'data'

IN_DIR = f'{DATA_DIR}/dataset'        # made by anchorage_dataset.ipynb
OUT_DIR = f'{DATA_DIR}/timeseries_v1' # made here
os.makedirs(OUT_DIR, exist_ok=True)

YEARS = range(2014, 2027)
table = pd.concat([pd.read_parquet(f'{IN_DIR}/cells_{y}.parquet') for y in YEARS], ignore_index=True)
info = pd.read_csv(f'{IN_DIR}/info.csv', parse_dates=['date'])

saved = np.load(f'{IN_DIR}/images_10m.npz')
images, row0, col0 = saved['images'], int(saved['row0']), int(saved['col0'])

print(f'{len(table):,} rows in the table, {len(info)} surveys, images {images.shape}')

## 2. Seasons

Between the last fall survey (October) and the first spring one (April/May) there are ~180 days,
and over that winter the whole harbor changes: typically 0.4–3.2 ft deeper.
Within a season, surveys are about a week apart.

So each summer is its own series: a new series starts wherever the gap to the previous survey is over 60 days.
`prev_id` is the survey before, within the same season (none for the first survey of a season).

Which surveys:
- all summer surveys (`IS`)
- the spring **pre-dredge survey** (kind `other`, late April/early May, most years from 2016): a full map just
  before the dredging season, so a clean first map for its season
- 2020-08-31 has two summer surveys on the same day: we keep one

In [ ]:
spring = (info.kind == 'other') & info.date.dt.month.isin([4, 5])
summer = info[(info.kind == 'IS') | spring].sort_values('date')
summer = summer.drop_duplicates('date', keep='last').reset_index(drop=True)

summer['gap_days'] = summer.date.diff().dt.days
summer['series'] = (summer.gap_days.isna() | (summer.gap_days > 60)).cumsum()
summer['prev_id'] = summer.groupby('series').survey_id.shift()

print(summer.series.nunique(), 'seasons')
summer.groupby('series').agg(start=('date', 'min'), end=('date', 'max'), surveys=('survey_id', 'size'))

## 3. Flag dredging

Most of the channel floor sits at **38–42 ft** (the dredged depth). Dredging removes shoals that rose above it
and puts them back at the floor. A cell counts as dredged since the previous survey when:

- it is more than **2 ft deeper** (the noise between surveys is ~0.4 ft)
- it was **shallower than 38 ft** before (a shoal)
- it is **34–46 ft** after (back at the floor)
- it is part of a patch of at least **20 connected cells** (dredging covers an area, not single cells)

Deepening in water that was already deep is not flagged: that is natural scour by the currents, which the model should learn.

In [ ]:
DEEPER = 2              # ft
SHOAL = 38              # ft, was shallower than this before
FLOOR = (34, 46)        # ft, is in this range after
MIN_PATCH = 20          # connected cells

def dredged_cells(before, after):
    """Compare two depth maps. Returns a True/False map of the cells that look dredged."""
    candidate = ((after - before > DEEPER) & (before < SHOAL)
                 & (after >= FLOOR[0]) & (after <= FLOOR[1]))     # comparisons with NaN give False

    patches, n = label(candidate)                  # number each group of touching cells 1, 2, 3, ...
    patch_size = np.bincount(patches.ravel())      # cells per patch number (index 0 = background)
    return (patches > 0) & (patch_size[patches] >= MIN_PATCH)

Run it on every summer survey that has a previous survey in its season.
The result `dredged` has the same shape as `images`.

In [ ]:
dredged = np.zeros(images.shape, dtype=bool)
for _, s in summer.dropna(subset=['prev_id']).iterrows():
    dredged[s.survey_id] = dredged_cells(images[int(s.prev_id)], images[s.survey_id])

summer['n_dredged'] = dredged[summer.survey_id].sum(axis=(1, 2))
flagged = summer[summer.n_dredged > 0]
print(f'{len(flagged)} of {summer.prev_id.notna().sum()} surveys have dredged cells')
print('dredged cells per flagged survey: median', flagged.n_dredged.median(), ', max', flagged.n_dredged.max())

Into the long table: each row looks up its own cell in the `dredged` maps, the same way step 8 of the dataset notebook
placed the table into images, just in the other direction.

In [ ]:
table['dredged'] = dredged[table.survey_id, table.row - row0, table.col - col0]
table.dredged.value_counts()

When does it happen? We expect a summer dredging season.

In [ ]:
flagged.groupby(flagged.date.dt.month).size().rename('flagged surveys').to_frame().T

## 4. Check the flags on maps

For a few flagged surveys: depth before, depth after, and the change, with the flagged cells outlined in red.
If the rule works, the red outlines sit on shallow spots (shoals) in the channel that got cut back to the floor.

In [ ]:
def show_pair(survey_id):
    s = summer.set_index('survey_id').loc[survey_id]
    before, after = images[int(s.prev_id)], images[survey_id]
    rows, cols = np.where(~np.isnan(after))
    crop = np.s_[rows.min():rows.max() + 1, cols.min():cols.max() + 1]   # zoom to where there is data

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    panels = [(before[crop], 'before', 'viridis_r', 20, 60),
              (after[crop], 'after', 'viridis_r', 20, 60),
              ((after - before)[crop], 'change (+ = deeper)', 'RdBu_r', -5, 5)]
    for ax, (img, title, cmap, vmin, vmax) in zip(axes, panels):
        im = ax.imshow(img, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax)
        ax.contour(dredged[survey_id][crop], levels=[0.5], colors='red', linewidths=1)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.6, label='ft')
    fig.suptitle(f'{s.date.date()}  ({int(s.gap_days)} days after the previous survey, {s.n_dredged} cells flagged)')
    plt.show()

In [ ]:
# the three surveys with the most flagged cells, and one typical one
examples = list(flagged.nlargest(3, 'n_dredged').survey_id)
examples.append(flagged[flagged.n_dredged.between(30, 50)].survey_id.iloc[0])
for survey_id in examples:
    show_pair(survey_id)

## 5. Area: crop to the core channel

`coverage` = for each cell, the share of our surveys that measured it.
About 5,900 cells are measured by nearly every survey: that is the channel the Corps surveys every week.
Around it, ~17,000 more cells are measured only now and then (edges, extra areas). A time series needs
the same cells every time, so the **core** is the cells covered by at least 80% of surveys
(80% or 95% gives almost the same area: the edge is sharp).

We crop all maps to the box around the core. Inside the box, cells outside the core are ignored via the mask (step 6).

In [ ]:
covered = ~np.isnan(images[summer.survey_id])     # (surveys, rows, cols), True where measured
coverage = covered.mean(axis=0)
core = coverage >= 0.8

rows, cols = np.where(core)
crop = np.s_[rows.min():rows.max() + 1, cols.min():cols.max() + 1]
print(f'core: {core.sum()} cells, box {rows.max() - rows.min() + 1} x {cols.max() - cols.min() + 1}')

plt.figure(figsize=(4, 7))
shown = plt.imshow(coverage, origin='lower', cmap='Greys')
plt.contour(core, levels=[0.5], colors='red', linewidths=1)
plt.colorbar(shown, label='share of surveys that cover the cell', shrink=0.6)
plt.title('coverage, core in red')
plt.show()

core = core[crop]

## 6. Build the samples

One sample = **K maps → a map further ahead**, all from the same season.

Why not simply the next survey? Surveys are only 3–4 days apart, and in that time a typical cell changes
~0.15 ft: mostly noise, and "nothing changes" is already nearly perfect. Over ~4 weeks the harbor shoals
~0.4 ft on average and ~30% of cells change by more than 1 ft. That is a forecast worth testing. So:

- `STEP = 2`: inputs are every 2nd survey (~1 week apart), `K = 4` of them (~3 weeks of history)
- `HORIZON = 8`: the target is 8 surveys (~4 weeks) after the last input

A season of 52 surveys gives 52 − (K−1)·STEP − HORIZON = 38 samples. Windows overlap; that is normal for time series.

We don't copy maps into every sample. `windows` only stores **which** maps make up each sample,
as positions in `maps` (the cropped maps of our surveys, in date order). `get_sample` assembles one when needed.

The **mask** says which cells of the target the model is graded on: cells in the core, measured in both the
last input and the target, and **not dredged anywhere between the first input and the target**
(including the surveys in between that the window skips).

In [ ]:
K = 4           # input maps
STEP = 2        # surveys between inputs
HORIZON = 8     # surveys from the last input to the target

maps = images[summer.survey_id][:, crop[0], crop[1]]            # (surveys, H, W), same order as `summer`
dredged_maps = dredged[summer.survey_id][:, crop[0], crop[1]]

windows = []
for _, season in summer.groupby('series'):
    positions = list(season.index)
    last_input = (K - 1) * STEP                                  # offset of the last input from the first
    for i in range(len(positions) - last_input - HORIZON):
        inputs = positions[i:i + last_input + 1:STEP]
        target = positions[i + last_input + HORIZON]
        windows.append(inputs + [target])                        # K inputs, then the target
windows = np.array(windows)

print('maps', maps.shape, '  windows', windows.shape)

In [ ]:
def get_sample(n):
    """Sample n: inputs (K, H, W), target (H, W), mask (H, W), and the days between consecutive maps (K)."""
    w = windows[n]
    inputs, target = maps[w[:-1]], maps[w[-1]]
    dredged_inside = dredged_maps[w[0] + 1:w[-1] + 1].any(axis=0)   # every survey from the first input to the target
    mask = core & ~np.isnan(inputs[-1]) & ~np.isnan(target) & ~dredged_inside
    gaps = np.diff(summer.date[w].values).astype('timedelta64[D]').astype(int)
    return inputs, target, mask, gaps

inputs, target, mask, gaps = get_sample(100)
print('inputs', inputs.shape, ' target', target.shape, ' graded cells', mask.sum(), ' gaps (days)', gaps)

Split by time, using the year of the target, so the model is always tested on years it has never seen:
train 2014–2022, validation 2023–2024, test 2025–2026.

In [ ]:
target_year = summer.date.dt.year.values[windows[:, -1]]
split = np.where(target_year <= 2022, 'train', np.where(target_year <= 2024, 'val', 'test'))

graded = np.array([get_sample(n)[2].sum() for n in range(len(windows))])
pd.DataFrame({'split': split, 'graded': graded}).groupby('split').graded.agg(samples='size', median_graded_cells='median')

One sample: the K inputs, the target, and the change from the last input to the target
(that change is what the model has to get right; "nothing changes" predicts 0 everywhere).

In [ ]:
n = 100
inputs, target, mask, gaps = get_sample(n)
w = windows[n]

fig, axes = plt.subplots(1, K + 2, figsize=(2.6 * (K + 2), 5))
for k in range(K):
    axes[k].imshow(inputs[k], origin='lower', cmap='viridis_r', vmin=20, vmax=60)
    axes[k].set_title(f'input {k + 1}\n{summer.date[w[k]].date()}', fontsize=9)
axes[K].imshow(target, origin='lower', cmap='viridis_r', vmin=20, vmax=60)
axes[K].set_title(f'target\n{summer.date[w[-1]].date()}', fontsize=9)
axes[K + 1].imshow(np.where(mask, target - inputs[-1], np.nan), origin='lower', cmap='RdBu_r', vmin=-3, vmax=3)
axes[K + 1].set_title('change (graded cells)\n+ = deeper', fontsize=9)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 7. Save (to `timeseries_v1/`)

- `timeseries.npz`: the cropped `maps`, `dredged` maps, `core`, `windows` and `split`, plus K, STEP, HORIZON,
  dates, survey ids and where the crop sits on the grid. Everything `get_sample` needs.
- `timeseries_surveys.csv`: the surveys used, with season, gap and number of dredged cells.
- `dredged_cells.parquet`: the `dredged` column of the long table, only the True rows (the rest is False).

In [ ]:
np.savez_compressed(f'{OUT_DIR}/timeseries.npz', maps=maps, dredged=dredged_maps, core=core,
                    windows=windows, split=split, K=K, STEP=STEP, HORIZON=HORIZON,
                    survey_id=summer.survey_id.values,
                    dates=summer.date.values.astype('datetime64[D]'),
                    row0=row0 + crop[0].start, col0=col0 + crop[1].start, cell=10)
summer.drop(columns='url').to_csv(f'{OUT_DIR}/timeseries_surveys.csv', index=False)
table[table.dredged][['survey_id', 'row', 'col']].to_parquet(f'{OUT_DIR}/dredged_cells.parquet')

print('saved', f'{OUT_DIR}/timeseries.npz', f'({os.path.getsize(f"{OUT_DIR}/timeseries.npz") / 1e6:.0f} MB)')